In [11]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Literal,Annotated
from langchain_core.messages import SystemMessage,HumanMessage,BaseMessage
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [12]:
load_dotenv()

True

In [13]:
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]#add_message is recommended for BaseMesage
    

In [14]:
model = ChatGroq(
    model="llama-3.1-8b-instant"  # free model
    
)

In [15]:
def chatnode(state:ChatState)->ChatState:
    messages=state['messages']
    response=model.invoke(messages)
    return {'messages': [response]}

In [16]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)

graph.add_node('chatnode',chatnode)
graph.add_edge(START,'chatnode')
graph.add_edge('chatnode',END)

chatbot=graph.compile(checkpointer=checkpointer)

In [9]:
initial_state={
    'messages':[HumanMessage(content="What is the capital of France?")]
}
chatbot.invoke(initial_state)

{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='018a5855-e917-4727-a36c-d43c72ea2587'),
  AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.005897949, 'completion_tokens_details': None, 'prompt_time': 0.005495417, 'prompt_tokens_details': None, 'queue_time': 0.053108833, 'total_time': 0.011393366}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0dda-86e8-7062-9aaa-691479c010f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})]}

In [17]:
thread_id='1'
while(True):
    user_input=input("Type here")
    print('User:',user_input)
    if user_input.strip().lower() in ['exit','quit','bye']:
        break
    response=chatbot.invoke({'messages':[HumanMessage(content=user_input)]}, config={'configurable': {'thread_id': thread_id}})
    print('Bot:',response['messages'][-1].content)

User: hello can you add 1 and 2
Bot: 1 + 2 = 3
User: now  multiply it with 4
Bot: 3 × 4 = 12
User: my name is abir
Bot: Nice to meet you, Abir. Is there anything else I can help you with?
User: tell me my name?
Bot: Your name is Abir.
User: bye
